In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
import rioxarray as rxr
import s3fs
import fsspec
from rasterio.warp import calculate_default_transform, reproject, Resampling
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")

✅ Libraries imported successfully!
Boto3 version: 1.37.3


In [2]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


In [3]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [4]:
EVENT_NAME = '202405_Heat_TX'  #find the name within drcs_activations OLD Directory (see link above)
PRODUCT_NAME = 'ECOSTRESS'      #find the name within drcs_activations OLD Directory (see link above)

RENAME_PRODUCT = 'ECOSTRESS'   #choose from LIST of 2nd level directories (see above list)

PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory
DIRECTORY_NEW = f'{DIR_NEW_BASE}/{RENAME_PRODUCT}'

In [5]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

⚠️ S3 client initialized (limited bucket list access)
✅ Confirmed access to nasa-disasters bucket
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 91 .tif files in the S3 bucket.


['drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023126133547_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023127043620_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023127043712_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023131025921_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023131030013_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023134021109_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023135012241_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001

In [6]:
# For simplicity, let's use python list comprehension to return the files
# We may need to rename them in different ways for different products
# We will do a similar process later

## NOTE --- We can actually use these objects since they have the same path as the s3 files. We will call them again later

lst = [f for f in keys if "_LST_" in f]
lsterr = [f for f in keys if "_LST_err_" in f]
qc = [f for f in keys if "_QC_" in f]

In [7]:
config_lst = {
    "data_acquisition_method": "s3",
    "raw_data_bucket" : BUCKET, #DO NOT CHANGE
    "raw_data_prefix": PATH_OLD,
    "cog_data_bucket": BUCKET, #DO NOT CHANGE
    "cog_data_prefix": f"{DIRECTORY_NEW}/LST",  #We changed this!!!!!!!
    "local_output_dir": f"output/{EVENT_NAME}",  # Local directory to save COGs
    "transformation": {}
}

config_lsterr = {
    "data_acquisition_method": "s3",
    "raw_data_bucket" : BUCKET, #DO NOT CHANGE
    "raw_data_prefix": PATH_OLD,
    "cog_data_bucket": BUCKET, #DO NOT CHANGE
    "cog_data_prefix": f"{DIRECTORY_NEW}/LST_err", #We changed this!!!!!!!
    "local_output_dir": f"output/{EVENT_NAME}",  # Local directory to save COGs
    "transformation": {}
}

config_qc = {
    "data_acquisition_method": "s3",
    "raw_data_bucket" : BUCKET, #DO NOT CHANGE
    "raw_data_prefix": PATH_OLD,
    "cog_data_bucket": BUCKET, #DO NOT CHANGE
    "cog_data_prefix": f"{DIRECTORY_NEW}/QC", #We changed this!!!!!!!
    "local_output_dir": f"output/{EVENT_NAME}",  # Local directory to save COGs
    "transformation": {}
}

In [8]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }




In [9]:
# Use the function with config_WM
lst_bucket = return_bucket_info(config_lst)
lsterr_bucket = return_bucket_info(config_lsterr)
qc_bucket = return_bucket_info(config_qc)

Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202405_Heat_TX/ECOSTRESS
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/ECOSTRESS/LST
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202405_Heat_TX/ECOSTRESS
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/ECOSTRESS/LST_err
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202405_Heat_TX/ECOSTRESS
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/ECOSTRESS/QC


In [11]:
def convert_ecostress_datetime(doy_str: str) -> str:
    """
    Convert ECOSTRESS timestamp 'YYYYJJJHHMMSS' (e.g., '2024136083721')
    to ISO 8601 UTC 'YYYY-MM-DDTHH:MM:SSZ'.

    Args:
        doy_str: String like '2024136083721' (year + DOY + hhmmss)

    Returns:
        ISO 8601 string like '2024-05-15T08:37:21Z'
    """
    from datetime import timedelta
    s = str(doy_str)
    if len(s) != 13 or not s.isdigit():
        raise ValueError(f"Expected 13-digit YYYYJJJHHMMSS, got: {doy_str}")

    year  = int(s[0:4])
    jday  = int(s[4:7])   # 001..366
    hour  = int(s[7:9])
    minute= int(s[9:11])
    second= int(s[11:13])

    # Convert year + day-of-year to calendar date, then add time
    base = datetime(year, 1, 1) + timedelta(days=jday - 1)
    dt   = base.replace(hour=hour, minute=minute, second=second)
    return dt.strftime("%Y-%m-%dT%H:%M:%SZ")

# Test
print(convert_ecostress_datetime('2024136083721'))  # 2024-05-15T08:37:21Z

2024-05-15T08:37:21Z


In [12]:
# Define COG profile for rasterio
COG_PROFILE = {
    "driver": "COG",
    "compress": "DEFLATE",
}

In [13]:
def convert_to_proper_CRS_and_cogify(name, cog_filename, cog_data_bucket, cog_data_prefix, local_output_dir=None):
    """
    Convert a file to Cloud Optimized GeoTIFF with proper CRS.
    
    This function includes:
    - Download caching to avoid re-downloading files
    - CRS reprojection to EPSG:4326
    - COG validation before upload
    - Upload to S3
    - Smart nodata value handling based on data type
    """
    s3_key = f"{cog_data_prefix}/{cog_filename}"
    reproject_filename = f"reproj/{cog_filename}"
    
    # Create necessary directories
    os.makedirs("reproj", exist_ok=True)
    
    # Create data_download directory for caching
    data_download_dir = "data_download"
    os.makedirs(data_download_dir, exist_ok=True)
    
    # Create subdirectory structure to match S3 path
    s3_path_parts = name.split('/')
    local_subdir = os.path.join(data_download_dir, *s3_path_parts[:-1])
    os.makedirs(local_subdir, exist_ok=True)
    
    # Local path for the downloaded file (persistent storage)
    local_download_path = os.path.join(data_download_dir, name)
    
    # Temporary file for processing
    temp_input_file = f"temp_{os.path.basename(name)}"

    try:
        # Check if file already exists locally
        if os.path.exists(local_download_path):
            print(f"   [CACHE HIT] Using cached file: {local_download_path}")
            import shutil
            shutil.copy(local_download_path, temp_input_file)
        else:
            # Download the file from S3
            print(f"   [DOWNLOAD] Downloading from S3...")
            s3_client.download_file(BUCKET, name, local_download_path)
            print(f"   [DOWNLOAD] ✅ Saved to cache")
            import shutil
            shutil.copy(local_download_path, temp_input_file)
        
        # Reproject to EPSG:4326
        print(f"   [REPROJECT] Converting to EPSG:4326...")
        with rasterio.open(temp_input_file) as src:
            dst_crs = "EPSG:4326"
            
            # Check if reprojection is needed
            if src.crs and src.crs.to_string() == dst_crs:
                print(f"   [REPROJECT] Already in {dst_crs}, skipping reprojection")
                import shutil
                shutil.copy(temp_input_file, reproject_filename)
            else:
                transform, width, height = calculate_default_transform(
                    src.crs, dst_crs, src.width, src.height, *src.bounds
                )
                kwargs = src.meta.copy()
                kwargs.update({
                    "driver": "COG",
                    "compress": "DEFLATE",
                    "crs": dst_crs,
                    "transform": transform,
                    "width": width,
                    "height": height
                })

                with rasterio.open(reproject_filename, "w", **kwargs) as dst:
                    for band_idx in range(1, src.count + 1):
                        reproject(
                            source=rasterio.band(src, band_idx),
                            destination=rasterio.band(dst, band_idx),
                            src_transform=src.transform,
                            src_crs=src.crs,
                            dst_transform=transform,
                            dst_crs=dst_crs,
                            resampling=Resampling.nearest,
                            wrapdateline=True
                        )

        # COGify & upload
        print(f"   [COGIFY] Creating COG...")
        ds = rxr.open_rasterio(reproject_filename)
        
        # Handle coordinate naming
        if "y" in ds.dims and "x" in ds.dims:
            ds = ds.rename({"y": "lat", "x": "lon"})
            ds.rio.set_spatial_dims("lon", "lat", inplace=True)
        
        # Smart nodata value handling based on data type
        print(f"   [NODATA] Data type: {ds.dtype}")
        if ds.dtype == 'uint8':
            # For RGB images (uint8), use 0 as nodata (black pixels)
            nodata_value = 0
            print(f"   [NODATA] Using nodata value {nodata_value} for uint8 data")
        elif ds.dtype == 'uint16':
            # For uint16, use 0 as nodata
            nodata_value = 0
            print(f"   [NODATA] Using nodata value {nodata_value} for uint16 data")
        else:
            # For float32, int16, int32, etc., use -9999
            nodata_value = -9999
            print(f"   [NODATA] Using nodata value {nodata_value} for {ds.dtype} data")
        
        ds.rio.write_nodata(nodata_value, inplace=True)

        with tempfile.NamedTemporaryFile(suffix='.tif', delete=False) as tmp:
            tmp_name = tmp.name
            ds.rio.to_raster(tmp_name, **COG_PROFILE)
            
            # Validate COG
            print(f"   [VALIDATE] Checking COG validity...")
            is_valid_cog, validation_details = validate_cog(tmp_name)
            
            if is_valid_cog:
                print(f"   [VALIDATE] ✅ Valid COG")
            else:
                print(f"   [VALIDATE] ⚠️ COG validation warnings")
                critical_errors = [e for e in validation_details['errors'] if 'Invalid driver' in e]
                if critical_errors:
                    raise ValueError(f"Critical COG validation failed")
                if 'errors' in validation_details:
                    for error in validation_details['errors']:
                        print(f"      - {error}")
                if 'warnings' in validation_details:
                    for warning in validation_details['warnings']:
                        print(f"      - {warning}")
            
            # Upload to S3
            print(f"   [UPLOAD] Uploading to S3...")
            s3_client.upload_file(
                Filename=tmp_name,
                Bucket=cog_data_bucket,
                Key=s3_key
            )
            print(f"   [SUCCESS] ✅ Uploaded to s3://{cog_data_bucket}/{s3_key}")
            
            # Save locally if specified
            if local_output_dir:
                os.makedirs(local_output_dir, exist_ok=True)
                local_path = os.path.join(local_output_dir, cog_filename)
                import shutil
                shutil.copy(tmp_name, local_path)
            
    except Exception as e:
        print(f"   [ERROR] Failed: {str(e)}")
        raise
            
    finally:
        # Clean up temporary files
        for temp_file in [temp_input_file, reproject_filename]:
            if os.path.exists(temp_file):
                os.remove(temp_file)
        if 'tmp_name' in locals() and os.path.exists(tmp_name):
            os.remove(tmp_name)

print("✅ COG conversion function defined with smart nodata handling")

✅ COG conversion function defined with smart nodata handling


In [14]:
# Check current cache status using the imported function
check_cache_status()

📊 Cache Status:
  - Directory: data_download/
  - Total files: 135
  - Total size: 4.91 GB

📁 Cached files (first 10):
  - drcs_activations/202405_Flood_TX/sentinel1/S1A_IW_20240430T002653_DVR_RTC20_G_gpuned_0610_WM.tif (3.1 MB)
  - drcs_activations/202405_Flood_TX/sentinel1/S1A_IW_20240430T002653_DVR_RTC20_G_gpuned_0610_rgb.tif (258.2 MB)
  - drcs_activations/202405_Flood_TX/sentinel1/S1A_IW_20240430T002719_DVR_RTC20_G_gpuned_F141_WM.tif (2.1 MB)
  - drcs_activations/202405_Flood_TX/sentinel1/S1A_IW_20240430T002719_DVR_RTC20_G_gpuned_F141_rgb.tif (289.2 MB)
  - drcs_activations/202405_Flood_TX/sentinel1/S1A_IW_20240507T122323_DVR_RTC20_G_gpuned_5BA0_WM.tif (2.7 MB)
  - drcs_activations/202405_Flood_TX/sentinel1/S1A_IW_20240507T122323_DVR_RTC20_G_gpuned_5BA0_rgb.tif (321.6 MB)
  - drcs_activations/202405_Flood_TX/sentinel1/S1A_IW_20240512T002655_DVR_RTC20_G_gpuned_EC9C_WM.tif (9.1 MB)
  - drcs_activations/202405_Flood_TX/sentinel1/S1A_IW_20240512T002720_DVR_RTC20_G_gpuned_D32B_WM.tif (

(135, 5273546571)

In [15]:

# Define filename creator functions for ECOSTRESS file types (robust to token order)
import re
from pathlib import Path

_DOY_13 = re.compile(r"doy(\d{13})", re.IGNORECASE)

def _extract_doy13_from_name(f: str) -> str:
    name = Path(f).stem
    m = _DOY_13.search(name)
    if not m:
        raise ValueError(f"No YYYYJJJHHMMSS after 'doy' in: {name}")
    return m.group(1)

def create_cog_filename_LST(f, EVENT_NAME):
    """Create COG filename for LST files."""
    ts_iso = convert_ecostress_datetime(_extract_doy13_from_name(f))
    return f"{EVENT_NAME}_ECOSTRESS_LST_{ts_iso}.tif"

def create_cog_filename_QC(f, EVENT_NAME):
    """Create COG filename for QC files."""
    ts_iso = convert_ecostress_datetime(_extract_doy13_from_name(f))
    return f"{EVENT_NAME}_ECOSTRESS_QC_{ts_iso}.tif"

def create_cog_filename_LST_ERR(f, EVENT_NAME):
    """Create COG filename for LST_err files."""
    ts_iso = convert_ecostress_datetime(_extract_doy13_from_name(f))
    return f"{EVENT_NAME}_ECOSTRESS_LSTERR_{ts_iso}.tif"


In [16]:
## Process files using batch processing function

print("📊 File categorization:")
print(f"  - LST files: {len(lst)}")
print(f"  - QC files: {len(qc)}")
print(f"  - LST ERR files: {len(lsterr)}")
print(f"  - Total files: {len(keys)}")



# Process water mask files
if lst:
    print("\n" + "="*50)
    print("Processing LST Files")
    print("="*50)
    # Initialize combined results DataFrame
    all_files_processed = pd.DataFrame()
    
    lst_results = process_file_batch(
        file_list=lst,
        s3_client=s3_client,
        config=config_lst,
        filename_creator_func=create_cog_filename_LST,
        processing_func=convert_to_proper_CRS_and_cogify,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True
    )
    all_files_processed = pd.concat([all_files_processed, lst_results], ignore_index=True)
    
    # Print overall summary
    print_batch_summary(all_files_processed)


# Process QC files
if qc:
    print("\n" + "="*50)
    print("Processing QC Files")
    print("="*50)
    # Initialize combined results DataFrame
    all_files_processed = pd.DataFrame()
    
    qc_results = process_file_batch(
        file_list=qc,
        s3_client=s3_client,
        config=config_qc,
        filename_creator_func=create_cog_filename_QC,
        processing_func=convert_to_proper_CRS_and_cogify,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True
    )
    all_files_processed = pd.concat([all_files_processed, qc_results], ignore_index=True)
    
    # Print overall summary
    print_batch_summary(all_files_processed)
    
# Process LST Error files
if lsterr:
    print("\n" + "="*50)
    print("Processing LST Err Files")
    print("="*50)
    # Initialize combined results DataFrame
    all_files_processed = pd.DataFrame()
    
    lsterr_results = process_file_batch(
        file_list=lsterr,
        s3_client=s3_client,
        config=config_lsterr,
        filename_creator_func=create_cog_filename_LST_ERR,
        processing_func=convert_to_proper_CRS_and_cogify,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True
    )
    all_files_processed = pd.concat([all_files_processed, lsterr_results], ignore_index=True)

    # Print overall summary
    print_batch_summary(all_files_processed)

📊 File categorization:
  - LST files: 60
  - QC files: 30
  - LST ERR files: 31
  - Total files: 91

Processing LST Files
✅ Local output directory ready: output/202405_Heat_TX

[1/60] Processing: drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023126133547_aid0001.tif
   Output filename: 202405_Heat_TX_ECOSTRESS_LST_2023-05-06T13:35:47Z.tif
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326...
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [COGIFY] Creating COG...
   [NODATA] Data type: uint16
   [NODATA] Using nodata value 0 for uint16 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/LST/202405_Heat_TX_ECOSTRESS_LST_2023-05-06T13:35:47Z.tif
   ✅ Generated and saved

In [ ]:
import os
os.getcwd()